In [21]:
import pandas as pd
import numpy as np

customers = pd.read_csv("Client data/customers.csv")
products = pd.read_csv("Client data/products.csv")
ratings = pd.read_csv("Client data/ratings.csv")
orders = pd.read_csv("Client data/orders.csv")


# Convert 'order_date' column to order datetime format
orders['order_date'] = pd.to_datetime(orders['order_date'])

In [3]:
# merge datasets

# merge_data = orders.merge(products, on='product_id').merge(customers, on='customer_id')
# merge_data

# Step 2: Prepare the data

#### Convert dates and calculate revenue per order.

In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Merge orders with products to get price information
sales = orders.merge(
    products[['product_id', 'product_name', 'price', 'category']],
    on='product_id',
    how='left'
)

# Revenue per transaction
sales['revenue'] = sales['quantity'] * sales['price']

# Top Performing Products by Revenue

In [5]:
top_products_revenue = (
    sales.groupby(['product_id', 'product_name'])
         .agg(
             total_revenue=('revenue', 'sum'),
             total_units=('quantity', 'sum')
         )
         .sort_values('total_revenue', ascending=False)
         .reset_index()
)

print(top_products_revenue.head(10))

   product_id         product_name  total_revenue  total_units
0           6  Brimnes Bed Storage          59521           77
1          42     Småstad Wardrobe          51688           52
2          29         Råskog Stool          50445           59
3          26        Docksta Table          46386           54
4          31         Nockeby Sofa          45646           58
5           5        Hemnes Daybed          44785           53
6          34   Fjälla Storage Box          43384           44
7          48       Bestå TV Bench          42320           46
8           1          Ektorp Sofa          40946           59
9          46   Valje Wall Cabinet          40140           60


# Top Performing Products by Units Sold

In [6]:
top_products_units = (
    sales.groupby(['product_id', 'product_name'])
         .agg(
             total_units=('quantity', 'sum'),
             total_revenue=('revenue', 'sum')
         )
         .sort_values('total_units', ascending=False)
         .reset_index()
)

print(top_products_units.head(10))

   product_id             product_name  total_units  total_revenue
0          39    Mackapar Shoe Storage           80          36640
1           6      Brimnes Bed Storage           77          59521
2          49   Söderhamn Sofa Section           63          27279
3           2           Poäng Armchair           63          35847
4          38  Bekant Conference Table           62          27342
5           8    Melltorp Dining Table           62          22878
6          30     Strandmon Wing Chair           61           5002
7          46       Valje Wall Cabinet           60          40140
8           1              Ektorp Sofa           59          40946
9          22     Nordli Chest Drawers           59          33099


# Identify Top Clients for the Last Month

#### Determine the most recent month in the data.

In [7]:
latest_date = sales['order_date'].max()

last_month_sales = sales[
    sales['order_date'] >= (latest_date - pd.DateOffset(months=1))
]

#### Join customer information.

In [8]:
last_month_sales = last_month_sales.merge(
    customers,
    on='customer_id',
    how='left'
)

#### Top Clients by Revenue

In [9]:
top_clients_revenue = (
    last_month_sales.groupby(['customer_id', 'name'])
                    .agg(
                        total_revenue=('revenue', 'sum'),
                        total_orders=('order_id', 'count'),
                        total_units=('quantity', 'sum')
                    )
                    .sort_values('total_revenue', ascending=False)
                    .reset_index()
)

print(top_clients_revenue.head(10))

   customer_id         name  total_revenue  total_orders  total_units
0            1   Customer_1          19755            12           29
1           33  Customer_33          17302            10           24
2            4   Customer_4          16562            13           34
3           81  Customer_81          15147             9           25
4           52  Customer_52          14939            11           25
5           50  Customer_50          14807             9           22
6           44  Customer_44          14473             7           20
7           54  Customer_54          14367            10           22
8           18  Customer_18          14356             8           23
9            8   Customer_8          14156             8           25


#### Top Clients by Quantity Purchased

In [10]:
top_clients_units = (
    last_month_sales.groupby(['customer_id', 'name'])
                    .agg(
                        total_units=('quantity', 'sum'),
                        total_revenue=('revenue', 'sum')
                    )
                    .sort_values('total_units', ascending=False)
                    .reset_index()
)

print(top_clients_units.head(10))

   customer_id         name  total_units  total_revenue
0            4   Customer_4           34          16562
1            1   Customer_1           29          19755
2           81  Customer_81           25          15147
3            8   Customer_8           25          14156
4           52  Customer_52           25          14939
5           72  Customer_72           24          14045
6           47  Customer_47           24          14146
7           33  Customer_33           24          17302
8           18  Customer_18           23          14356
9           21  Customer_21           23           8430


#### Bonus: Include Product Ratings

In [11]:
# to see whether highly-rated products are also top sellers

avg_ratings = (
    ratings.groupby('product_id')
           .agg(avg_rating=('rating', 'mean'))
           .reset_index()
)

product_performance = (
    top_products_revenue.merge(
        avg_ratings,
        on='product_id',
        how='left'
    )
)

print(product_performance.head(10))

   product_id         product_name  total_revenue  total_units  avg_rating
0           6  Brimnes Bed Storage          59521           77    3.636364
1          42     Småstad Wardrobe          51688           52    2.714286
2          29         Råskog Stool          50445           59    3.428571
3          26        Docksta Table          46386           54    2.857143
4          31         Nockeby Sofa          45646           58    3.000000
5           5        Hemnes Daybed          44785           53    2.500000
6          34   Fjälla Storage Box          43384           44    2.750000
7          48       Bestå TV Bench          42320           46    3.100000
8           1          Ektorp Sofa          40946           59    2.800000
9          46   Valje Wall Cabinet          40140           60    3.000000


# Executive Summary Table

#### A useful dashboard-style summary:

In [12]:
summary = (
    sales.groupby(['product_id', 'product_name', 'category'])
         .agg(
             total_revenue=('revenue', 'sum'),
             total_units=('quantity', 'sum')
         )
         .reset_index()
)

summary = summary.merge(
    ratings.groupby('product_id')['rating']
           .mean()
           .reset_index()
           .rename(columns={'rating': 'avg_rating'}),
    on='product_id',
    how='left'
)

summary = summary.sort_values(
    'total_revenue',
    ascending=False
)

print(summary.head(20))

    product_id           product_name           category  total_revenue  \
5            6    Brimnes Bed Storage               Beds          59521   
41          42       Småstad Wardrobe  Storage Solutions          51688   
28          29           Råskog Stool             Chairs          50445   
25          26          Docksta Table     Tables & Desks          46386   
30          31           Nockeby Sofa  Sofas & Armchairs          45646   
4            5          Hemnes Daybed               Beds          44785   
33          34     Fjälla Storage Box  Storage Solutions          43384   
47          48         Bestå TV Bench              Decor          42320   
0            1            Ektorp Sofa  Sofas & Armchairs          40946   
45          46     Valje Wall Cabinet  Storage Solutions          40140   
36          37            Fredde Desk     Tables & Desks          39422   
26          27           Ivar Cabinet  Storage Solutions          37785   
9           10   Kallax S

# Customer Segmentation (Highest Priority)

Not all customers are equally valuable.

Questions to answer
<li>Who are the biggest spenders?</li>
<li>Who buys frequently?</li>
<li>Who buys only once?</li>
<li>Which customers are at risk of never returning?</li>

#### Metrics

In [13]:
customer_metrics = (
    sales.groupby('customer_id')
         .agg(
             total_spend=('revenue','sum'),
             total_orders=('order_id','nunique'),
             total_units=('quantity','sum'),
             first_purchase=('order_date','min'),
             last_purchase=('order_date','max')
         )
)

In [14]:
customer_metrics

,total_spend,total_orders,total_units,first_purchase,last_purchase
customer_id,,,,,
1,27081,18,39,2023-07-21 08:39:23.971834,2023-09-17 08:39:23.971834
2,9347,9,21,2023-07-24 08:39:23.971834,2023-09-15 08:39:23.971834
3,18077,13,33,2023-07-20 08:39:23.971834,2023-09-13 08:39:23.971834
4,25717,22,55,2023-07-22 08:39:23.971834,2023-09-14 08:39:23.971834
5,14913,11,24,2023-07-20 08:39:23.971834,2023-09-09 08:39:23.971834
...,...,...,...,...,...
96,12264,9,24,2023-07-24 08:39:23.971834,2023-09-15 08:39:23.971834
97,6088,4,9,2023-07-21 08:39:23.971834,2023-09-09 08:39:23.971834
98,8915,7,17,2023-07-21 08:39:23.971834,2023-09-11 08:39:23.971834


In [15]:
# Ensure date is datetime
sales['order_date'] = pd.to_datetime(sales['order_date'])

# Add useful date columns
sales['year_month'] = sales['order_date'].dt.to_period('M')
sales['week'] = sales['order_date'].dt.to_period('W')
sales['day'] = sales['order_date'].dt.date

# 1. Customer Segmentation

In [16]:
customer_segmentation = (
    sales.groupby('customer_id')
    .agg(
        total_spend=('revenue', 'sum'),
        total_orders=('order_id', 'nunique'),
        total_units=('quantity', 'sum'),
        avg_order_value=('revenue', 'mean'),
        first_purchase=('order_date', 'min'),
        last_purchase=('order_date', 'max')
    )
    .reset_index()
)

customer_segmentation = customer_segmentation.merge(
    customers,
    on='customer_id',
    how='left'
)

customer_segmentation.sort_values('total_spend', ascending=False).head(10)

,customer_id,total_spend,total_orders,total_units,avg_order_value,first_purchase,last_purchase,name
0,1,27081,18,39,1504.500000,2023-07-21 08:39:23.971834,2023-09-17 08:39:23.971834,Customer_1
43,44,26529,17,41,1560.529412,2023-07-21 08:39:23.971834,2023-09-14 08:39:23.971834,Customer_44
3,4,25717,22,55,1168.954545,2023-07-22 08:39:23.971834,2023-09-14 08:39:23.971834,Customer_4
28,29,24997,14,37,1785.500000,2023-07-23 08:39:23.971834,2023-09-17 08:39:23.971834,Customer_29
51,52,24055,16,42,1503.437500,2023-07-24 08:39:23.971834,2023-09-17 08:39:23.971834,Customer_52
91,92,22082,17,39,1298.941176,2023-07-21 08:39:23.971834,2023-09-12 08:39:23.971834,Customer_92
54,55,21599,11,29,1963.545455,2023-07-26 08:39:23.971834,2023-09-17 08:39:23.971834,Customer_55
32,33,21387,15,36,1425.800000,2023-07-28 08:39:23.971834,2023-09-16 08:39:23.971834,Customer_33
69,70,21089,13,34,1622.230769,2023-07-20 08:39:23.971834,2023-09-17 08:39:23.971834,Customer_70
99,100,20744,12,34,1728.666667,2023-07-30 08:39:23.971834,2023-09-06 08:39:23.971834,Customer_100


# 2. RFM Analysis

In [17]:
snapshot_date = sales['order_date'].max() + pd.Timedelta(days=1)

rfm = (
    sales.groupby('customer_id')
    .agg(
        recency=('order_date', lambda x: (snapshot_date - x.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('revenue', 'sum')
    )
    .reset_index()
)

rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])

rfm['RFM_score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm = rfm.merge(customers, on='customer_id', how='left')

rfm.sort_values('monetary', ascending=False).head(10)

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,name
0,1,1,18,27081,5,5,5,555,Customer_1
43,44,4,17,26529,4,5,5,455,Customer_44
3,4,4,22,25717,4,5,5,455,Customer_4
28,29,1,14,24997,5,5,5,555,Customer_29
51,52,1,16,24055,5,5,5,555,Customer_52
91,92,6,17,22082,3,5,5,355,Customer_92
54,55,1,11,21599,5,4,5,545,Customer_55
32,33,2,15,21387,5,5,5,555,Customer_33
69,70,1,13,21089,5,5,5,555,Customer_70
99,100,12,12,20744,1,4,5,145,Customer_100


# 3. Category Performance

In [19]:
category_performance = (
    sales.groupby('category')
    .agg(
        total_revenue=('revenue', 'sum'),
        total_units=('quantity', 'sum'),
        total_orders=('order_id', 'nunique')
    )
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

category_performance

,category,total_revenue,total_units,total_orders
0,Storage Solutions,329700,508,214
1,Decor,242454,379,156
2,Tables & Desks,220130,450,191
3,Sofas & Armchairs,203318,317,124
4,Chairs,123019,280,118
5,Beds,116702,208,89
6,Lighting,114844,271,108


# 4. Product Rating vs Revenue

In [22]:
avg_ratings = (
    ratings.groupby('product_id')
    .agg(avg_rating=('rating', 'mean'))
    .reset_index()
)

product_rating_revenue = (
    sales.groupby(['product_id', 'product_name', 'category'])
    .agg(
        total_revenue=('revenue', 'sum'),
        total_units=('quantity', 'sum')
    )
    .reset_index()
    .merge(avg_ratings, on='product_id', how='left')
)

product_rating_revenue['action'] = np.where(
    (product_rating_revenue['total_revenue'] >= product_rating_revenue['total_revenue'].median()) &
    (product_rating_revenue['avg_rating'] >= product_rating_revenue['avg_rating'].median()),
    'Promote aggressively',
    np.where(
        (product_rating_revenue['total_revenue'] >= product_rating_revenue['total_revenue'].median()) &
        (product_rating_revenue['avg_rating'] < product_rating_revenue['avg_rating'].median()),
        'Improve product quality',
        np.where(
            (product_rating_revenue['total_revenue'] < product_rating_revenue['total_revenue'].median()) &
            (product_rating_revenue['avg_rating'] >= product_rating_revenue['avg_rating'].median()),
            'Increase marketing',
            'Consider removing or discounting'
        )
    )
)

product_rating_revenue.sort_values('total_revenue', ascending=False).head(20)

,product_id,product_name,category,total_revenue,total_units,avg_rating,action
5,6,Brimnes Bed Storage,Beds,59521,77,3.636364,Promote aggressively
41,42,Småstad Wardrobe,Storage Solutions,51688,52,2.714286,Improve product quality
28,29,Råskog Stool,Chairs,50445,59,3.428571,Promote aggressively
25,26,Docksta Table,Tables & Desks,46386,54,2.857143,Improve product quality
30,31,Nockeby Sofa,Sofas & Armchairs,45646,58,3.000000,Promote aggressively
4,5,Hemnes Daybed,Beds,44785,53,2.500000,Improve product quality
33,34,Fjälla Storage Box,Storage Solutions,43384,44,2.750000,Improve product quality
47,48,Bestå TV Bench,Decor,42320,46,3.100000,Promote aggressively
0,1,Ektorp Sofa,Sofas & Armchairs,40946,59,2.800000,Improve product quality
45,46,Valje Wall Cabinet,Storage Solutions,40140,60,3.000000,Promote aggressively


# 5. Average Order Value

In [23]:
order_values = (
    sales.groupby('order_id')
    .agg(order_revenue=('revenue', 'sum'))
    .reset_index()
)

average_order_value = order_values['order_revenue'].mean()

print("Average Order Value:", round(average_order_value, 2))

Average Order Value: 1350.17


# 6. Monthly Revenue Trend

In [24]:
monthly_revenue = (
    sales.groupby('year_month')
    .agg(
        total_revenue=('revenue', 'sum'),
        total_orders=('order_id', 'nunique'),
        total_units=('quantity', 'sum')
    )
    .reset_index()
)

monthly_revenue['year_month'] = monthly_revenue['year_month'].astype(str)

monthly_revenue

,year_month,total_revenue,total_orders,total_units
0,2023-07,278127,193,487
1,2023-08,691642,530,1270
2,2023-09,380398,277,656


# 7. Repeat Purchase Analysis

In [25]:
repeat_purchase = (
    sales.groupby('customer_id')
    .agg(total_orders=('order_id', 'nunique'))
    .reset_index()
)

repeat_purchase['customer_type'] = np.where(
    repeat_purchase['total_orders'] > 1,
    'Repeat Customer',
    'One-Time Customer'
)

repeat_summary = (
    repeat_purchase['customer_type']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

repeat_summary.columns = ['customer_type', 'percentage']

repeat_summary

,customer_type,percentage
0,Repeat Customer,100.0


# 8. Churn Analysis

In [26]:
latest_date = sales['order_date'].max()
churn_days = 60

customer_last_purchase = (
    sales.groupby('customer_id')
    .agg(last_purchase=('order_date', 'max'))
    .reset_index()
)

customer_last_purchase['days_since_last_purchase'] = (
    latest_date - customer_last_purchase['last_purchase']
).dt.days

customer_last_purchase['churn_status'] = np.where(
    customer_last_purchase['days_since_last_purchase'] > churn_days,
    'Churned',
    'Active'
)

customer_last_purchase = customer_last_purchase.merge(
    customers,
    on='customer_id',
    how='left'
)

customer_last_purchase.head()

,customer_id,last_purchase,days_since_last_purchase,churn_status,name
0,1,2023-09-17 08:39:23.971834,0,Active,Customer_1
1,2,2023-09-15 08:39:23.971834,2,Active,Customer_2
2,3,2023-09-13 08:39:23.971834,4,Active,Customer_3
3,4,2023-09-14 08:39:23.971834,3,Active,Customer_4
4,5,2023-09-09 08:39:23.971834,8,Active,Customer_5


# 9. Pareto Analysis: 80/20 Product Revenue

In [27]:
pareto_products = (
    sales.groupby(['product_id', 'product_name'])
    .agg(total_revenue=('revenue', 'sum'))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

pareto_products['revenue_percentage'] = (
    pareto_products['total_revenue'] / pareto_products['total_revenue'].sum()
) * 100

pareto_products['cumulative_percentage'] = pareto_products['revenue_percentage'].cumsum()

pareto_products.head(20)

,product_id,product_name,total_revenue,revenue_percentage,cumulative_percentage
0,6,Brimnes Bed Storage,59521,4.408418,4.408418
1,42,Småstad Wardrobe,51688,3.828267,8.236685
2,29,Råskog Stool,50445,3.736204,11.972889
3,26,Docksta Table,46386,3.435575,15.408464
4,31,Nockeby Sofa,45646,3.380767,18.789231
5,5,Hemnes Daybed,44785,3.316997,22.106228
6,34,Fjälla Storage Box,43384,3.213232,25.319460
7,48,Bestå TV Bench,42320,3.134427,28.453888
8,1,Ektorp Sofa,40946,3.032662,31.486549
9,46,Valje Wall Cabinet,40140,2.972966,34.459515


# 10. Market Basket Analysis

In [28]:
basket = (
    sales.groupby(['order_id', 'product_name'])['quantity']
    .sum()
    .unstack()
    .fillna(0)
)

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

product_combinations = basket.T.dot(basket)

np.fill_diagonal(product_combinations.values, 0)

product_combinations

C:\Users\manga\AppData\Local\Temp\ipykernel_14424\3726673919.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


product_name,Askvoll Wardrobe,Bekant Conference Table,Bernhard Chair,Bestå TV Bench,Billy Bookcase,Brimnes Bed Storage,Docksta Table,Ektorp Sofa,Fjälla Storage Box,Fredde Desk,...,Skurup Ceiling Lamp,Småstad Wardrobe,Stockholm Mirror,Strandmon Wing Chair,Stuva Loft Bed,Söderhamn Sofa Section,Tarva Nightstand,Terje Folding Chair,Tingby Side Table,Valje Wall Cabinet
product_name,,,,,,,,,,,,,,,,,,,,,
Askvoll Wardrobe,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Bekant Conference Table,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Bernhard Chair,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Bestå TV Bench,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Billy Bookcase,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Brimnes Bed Storage,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Docksta Table,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ektorp Sofa,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Fjälla Storage Box,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
# 11 Export Results to Excel

with pd.ExcelWriter("online_store_analysis.xlsx") as writer:
    customer_segmentation.to_excel(writer, sheet_name="Customer Segments", index=False)
    rfm.to_excel(writer, sheet_name="RFM Analysis", index=False)
    category_performance.to_excel(writer, sheet_name="Category Performance", index=False)
    product_rating_revenue.to_excel(writer, sheet_name="Product Rating Revenue", index=False)
    monthly_revenue.to_excel(writer, sheet_name="Monthly Revenue", index=False)
    repeat_summary.to_excel(writer, sheet_name="Repeat Purchase", index=False)
    customer_last_purchase.to_excel(writer, sheet_name="Churn Analysis", index=False)
    pareto_products.to_excel(writer, sheet_name="Pareto Products", index=False)
    

print("Analysis exported successfully.")

Analysis exported successfully.
